# Week 51, Day 6: Project report and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w051/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

This lesson uses a dataset from the Engram site. Run this cell first: it downloads the file once and defines `dataset()`.

In [ ]:
import os, urllib.request
def dataset(name):
    """Downloads an Engram dataset the first time it is used and returns its local path."""
    if not os.path.exists(name):
        urllib.request.urlretrieve("https://engram-neuroai.vercel.app/data/" + name, name)
    return name

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.signal import lfilter
from scipy.special import erf
d = np.load(dataset("ibl_biased.npz"))
mouse, session = d["mouse"].astype(int), d["session"].astype(int)
contrast, choice, reward = d["contrast"].astype(float), d["choice"].astype(float), d["reward"].astype(int)
p_left, rt = d["p_left"].astype(float).round(2), d["rt"].astype(float)
ok = (choice >= 0) & d["included"] & np.isfinite(rt)
sess = 10 * mouse + session
c_pm = np.where(choice >= 0, 2 * choice - 1, 0.0)
side = np.where(contrast != 0, np.sign(contrast), np.where(reward == 1, c_pm, -c_pm))

def kernel(x, alpha, sess_ids):
    out = np.zeros(len(x))
    for s in np.unique(sess_ids):
        k = sess_ids == s
        out[k] = lfilter([alpha], [1, -(1 - alpha)], np.r_[0, x[k][:-1]])
    return out

def bayes_prior(side, sess_ids, h=0.02):
    out = np.zeros(len(side))
    for s in np.unique(sess_ids):
        b = np.array([0.5, 0.5])
        for t in np.flatnonzero(sess_ids == s):
            pr = 0.8 * b[0] + 0.2 * b[1]; out[t] = np.log(pr / (1 - pr))
            b = b * (np.array([0.8, 0.2]) if side[t] > 0 else np.array([0.2, 0.8])); b = b / b.sum()
            b = np.array([(1 - h) * b[0] + h * b[1], (1 - h) * b[1] + h * b[0]])
    return out

models = ["bayes", "stimulus", "action"]
x0 = {"bayes": [3, 0, 1], "stimulus": [3, 0, 1, 0.2], "action": [3, 0, 1, 0.2]}
bounds = {"bayes": [(-20, 20)] * 3, "stimulus": [(-20, 20)] * 3 + [(0.01, 1)], "action": [(-20, 20)] * 3 + [(0.01, 1)]}
prior_cache = {0.02: bayes_prior(side, sess)}

def compare(scale=5, use=None, h=0.02):
    """-BIC/2 of the three models for every mouse, under a contrast transform tanh(scale c), a trial mask and a switch rate h."""
    use = ok if use is None else ok & use
    fc = np.tanh(scale * contrast) / np.tanh(scale)
    if h not in prior_cache:
        prior_cache[h] = bayes_prior(side, sess, h)
    pb = prior_cache[h]
    lme = np.zeros((18, 3))
    for m in range(18):
        k = mouse == m; u = use[k]; y = choice[k][u]; n = u.sum()
        for j, mod in enumerate(models):
            def f(p):
                z = p[0] * fc[k] + p[1]
                if mod == "bayes":
                    z = z + p[2] * pb[k]
                else:
                    z = z + p[2] * kernel(side[k] if mod == "stimulus" else c_pm[k], p[3], sess[k])
                z = z[u]
                return np.sum(np.logaddexp(0, z) - y * z)
            res = minimize(f, x0[mod], method="L-BFGS-B", bounds=bounds[mod])
            lme[m, j] = -res.fun - len(x0[mod]) * np.log(n) / 2
    return lme
print("ready")

### Exercise: Does the result depend on my choices?

Run compare for three variants: a gentler transform (scale 2), a harsher one (scale 10), and fitting only trials in biased blocks. For each, store in results the number of mice for which the action kernel wins and its summed lead over the next best model.

In [ ]:
variants = {"scale 2": dict(scale=2), "scale 10": dict(scale=10), "biased trials only": dict(use=p_left != 0.5)}
results = {}
for name, kw in variants.items():
    lme = compare(**kw)
    wins = ...
    tot = lme.sum(0)
    lead = ...
    results[name] = (wins, lead)
    print(f"{name:20s} action kernel best for {wins} of 18 mice, leading by {lead:.0f} nats")

In [ ]:
import numpy as np
def test_keys():
    "results holds the counts and margins for the three variants"
    assert set(results) == {"scale 2", "scale 10", "biased trials only"}
def test_counts():
    "in every variant the action kernel wins for at least 15 mice"
    assert all(v[0] >= 15 for v in results.values())
def test_margin():
    "in every variant the action kernel leads the next best model by more than 1,000 nats"
    assert all(v[1] > 1000 for v in results.values())

test_keys()
test_counts()
test_margin()
print("All checks passed")

### Exercise: The report's numbers

Complete project_summary: the zero-contrast choice rates in each block, the number of mice whose psychometric bias moves towards the likely side, and, from compare(), the number of mice for which the action kernel wins, the fraction of mice it wins for, and its summed lead over each rival.

In [ ]:
def psychometric(q, c):
    return q[2] + (1 - q[2] - q[3]) * 0.5 * (1 + erf((c - q[0]) / (q[1] * np.sqrt(2))))

def fit_mu(c, y):
    f = lambda q: -np.sum(y * np.log(np.clip(psychometric(q, c), 1e-9, 1 - 1e-9)) + (1 - y) * np.log(np.clip(1 - psychometric(q, c), 1e-9, 1 - 1e-9)))
    return minimize(f, [0, 0.1, 0.05, 0.05], method="L-BFGS-B", bounds=[(-0.5, 0.5), (0.005, 1), (0, 0.5), (0, 0.5)]).x[0]

def project_summary():
    z = ok & (contrast == 0)
    shifts = [fit_mu(contrast[ok & (mouse == m) & (p_left == 0.8)], choice[ok & (mouse == m) & (p_left == 0.8)])
              - fit_mu(contrast[ok & (mouse == m) & (p_left == 0.2)], choice[ok & (mouse == m) & (p_left == 0.2)]) for m in range(18)]
    lme = compare()
    tot = lme.sum(0)
    return {
        "n_trials": len(choice),
        "n_mice": len(np.unique(mouse)),
        "p_right_zero_rightblock": ...,
        "p_right_zero_leftblock": ...,
        "mice_shifted": ...,
        "action_wins": ...,
        "action_freq_simple": ...,
        "lead_over_stimulus": ...,
        "lead_over_bayes": ...,
    }

summary = project_summary()
for k, v in summary.items():
    print(f"{k:24s} {v:.3f}" if isinstance(v, float) else f"{k:24s} {v}")

In [ ]:
import numpy as np
def test_keys():
    "summary has the required keys"
    need = {"n_trials", "n_mice", "p_right_zero_rightblock", "p_right_zero_leftblock", "mice_shifted", "action_wins", "action_freq_simple", "lead_over_stimulus", "lead_over_bayes"}
    assert need <= set(summary)
def test_values():
    "the numbers match the week's results"
    assert summary["n_trials"] == 70402 and summary["n_mice"] == 18 and summary["mice_shifted"] == 18
    assert summary["action_wins"] == 17 and summary["lead_over_stimulus"] > 1500
    assert 0.55 < summary["p_right_zero_rightblock"] < 0.65 and 0.33 < summary["p_right_zero_leftblock"] < 0.42

test_keys()
test_values()
print("All checks passed")